[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg))](https://colab.research.google.com/github/NOAA-STAR/aerosolwatch-training/blob/main/colab_tutorials/tempo/tempo-abi_pm25_colab_tutorial.ipynb)

# **Google Colab Tutorial: TEMPO-ABI Hourly Estimated Surface PM2.5**

<font color='blue'>Author: Dr. Amy Huff (amy.huff@noaa.gov)</font>

This tutorial is designed to be run on [Google Colab](https://colab.google/). The tutorial demonstrates how to work with TEMPO-ABI hourly estimated surface PM2.5 data, featuring the Python `Xarray`, `cartopy`, and `Matplotlib` packages.

<font color='red'>**If you use any of the Python code in this tutorial, please credit the NOAA/NESDIS Aerosols & Atmospheric Composition Science Team.**</font>

---
**TUTORIAL OVERVIEW**

**Section 1 (Open a TEMPO-ABI PM2.5 File)**:
- Open a netCDF4 file that uses the `groups` organizational structure
- Identify data variables and review their metadata
- Read PM2.5 data arrays & assign them to variables

**Section 2 (Latitude & Longitude for TEMPO-ABI PM2.5 Files)**:
- Understand the fixed latitude & longitude grids for hourly satellite PM2.5 data
- Open a netCDF4 file containing interpolated (no missing values) latitude and longitude arrays
- Read latitude & longitude arrays & assign them to variables

**Section 3 (Get Hourly AirNow PM2.5 Data)**:
- Stream the AirNowTech data file containing hourly regulatory pollutant monitor observations for the same time as the satellite PM2.5 file
- Extract PM2.5 monitor data into a `pandas` Dataframe

**Section 4 (Generate ABI True Color RGB Composite using `SatPy`)**:
- Generate a true color RGB `NumPy` array and its projection information for the GOES-East ABI CONUS sector observation at ~30 minutes past the observation hour of the satellite & AirNow PM2.5 files.

**Section 5 (Plot PM2.5 Data on a Map)**:
- Create a color map for PM2.5 concentrations using `Matplotlib`
- Plot hourly TEMPO-ABI PM2.5 and AirNow PM2.5 data on a map, with the ABI true color RGB as a background
- Add enhancements to the map:
  - Borderlines
  - Colorbar
- Save map plot as an image (.png) file


<center><b><font size='4'>Case study event: TEMPO-ABI PM2.5 for 19 UTC on May 14, 2026, overlaid with AirNow monitor PM2.5</font></b></center>

<center><font size='3'>(background: GOES-East ABI true color RGB @ 19:31 UTC)</font></center>

<div align="center">
<img src="https://raw.githubusercontent.com/NOAA-STAR/aerosolwatch-training/main/images/tempo-abi_airnow_pm25_rgb_20260514_19.png" width="800">
</div>

## **Read the Satellite Data Documentation!**

Before working with a satellite product, users should read the product documentation from the science team, such as the users' guide, "readme", or algorithm theoretical basis document (ATBD). These documents provide important information about satellite products, including valid data ranges, data screening using data quality/diagnostic flags, and known issues.

Here is the link to the documentation for the satellite product used in this tutorial:

- [User's Guide for Estimated Surface PM2.5 from TEMPO & ABI Aerosol Retrievals](https://www.star.nesdis.noaa.gov/pub/smcd/akhuff/Product_Users_Guides/TEMPO-ABI_PM2.5_Users_Guide_V2.2_20260512.pdf) (May 2026)



## **Section 0: Set up Google Colab**

[Google Colab](https://colab.google/) is a hosted Jupyter Notebook-based service that requires no setup to use and provides free access to computing resources.

[Jupyter Notebook](https://jupyter-notebook.readthedocs.io/en/stable/notebook.html) is an open-source web application that supports >40 programming languages, including Python. It allows code to be broken into “blocks” that run independently, which makes it ideal for learning. Any output from the code in a "block" will appear underneath it.

**The Python code demonstrated in this training is universal**. Specific lines of code or functions will run in any Python IDE (e.g., Spyder, Visual Studio Code), Jupyter Notebook, or the Python interpreter.

###**Example of how to run Notebook code blocks**

To see how a Notebook works, let's run the Python code to print "Hello world!"

Place your cursor over the grey code block below, then click the little black circle with the white arrow inside, located on the far left side of the block.

In [ ]:
print('Hello world!')

### **Limitations of using Google Colab**

Colab is free, powerful, and easy to use, but it has limitations:


1.   **Colab sessions are temporary.** A session will expire after 12 hours of continuous use or after 90 minutes of idle time. <font color='red'>**All output, including downloaded or generated files, will be lost after the session expires.**</font> Therefore, any files users want to save must be downloaded to the user's local computer or Google Drive account.
2.  **Colab cannot be configured to use a virtual or conda environment.** Therefore, in general, users must work with the existing, current Colab configuration.
3. **The Colab configuration is updated every ~1 month, with the addition of new packages and updates to existing packages.** Consequently, code that runs today may give an error in the future after an update to the Colab configuration.

### **Import Python modules and packages**

- [pathlib](https://docs.python.org/3/library/pathlib.html): module to set file system paths
- [io](https://docs.python.org/3/library/io.html): module for working with I/O streams
- [requests](https://requests.readthedocs.io/en/latest/): library to send HTTP requests
- [Xarray](https://docs.xarray.dev/en/stable/index.html): library to work with labeled multi-dimensional arrays
- [NumPy](https://numpy.org/doc/stable/user/index.html): library to perform array operations
- [pandas](https://pandas.pydata.org/docs/index.html): library for data analysis
- [Matplotlib](https://matplotlib.org/stable/index.html): library to make plots
- [cartopy](https://scitools.org.uk/cartopy/docs/latest/index.html): library to make maps
- [SatPy](https://satpy.readthedocs.io/en/stable/): library designed to make working with satellite data easy

Used by `xarray` but not imported:
- [netCDF4](https://unidata.github.io/netcdf4-python/): library to read and write netCDF files

Used by `SatPy` but not imported:
- [PySpectral](https://pyspectral.readthedocs.io/en/latest/): library to read and manipulate satellite sensor spectral responses and solar irradiance spectra
---
---
The Colab configuration does not include the `cartopy`, `netcdf4`, `satpy` or `pyspectral` packages, so they need to be installed.

**Ignore any error messages about package dependency conflicts; they will not impact the training.**

In [ ]:
# Install missing packages in Colab quietly (no progress notifications)
!pip install --quiet cartopy netcdf4 satpy pyspectral

In [ ]:
# Import modules
from pathlib import Path
from io import StringIO

# Import packages
import requests
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from satpy import Scene
from satpy.enhancements.enhancer import get_enhanced_image

##**Section 1: Open a TEMPO-ABI PM2.5 File**

###**Upload TEMPO-ABI PM2.5 data file**

TEMPO aerosol products are not yet operational, so recent data files are not available publicly.

For convenience for this tutorial, the TEMPO-ABI PM2.5 data file for May 14, 2026 at 19 UTC can be uploaded to the Colab instance from my NOAA public web archive.

In [ ]:
# Query NOAA web archive for satellite data files
data_url = ('https://www.star.nesdis.noaa.gov/pub/smcd/akhuff/'
            'Tutorial_Satellite_Data_Files/'
            'TEMPO-ABI_PM25_L4_V02_20260514T190000Z.nc')
response = requests.get(data_url,
                        timeout=(5,20))

# Set directory path to file
pm25_file = Path.cwd() / 'TEMPO-ABI_PM25_L4_V02_20260514T190000Z.nc'

# Save file to Colab instance
with open(pm25_file, 'wb') as file:
    file.write(response.content)

###**Open TEMPO-ABI PM2.5 file as an `xarray` `DataTree`**

All TEMPO [netCDF4](https://docs.unidata.ucar.edu/netcdf-c/current/netcdf_data_model.html) data files use the [groups](https://archive.unidata.ucar.edu/software/netcdf/workshops/2011/groups-types/GroupsIntro.html) organizational structure. In order to read `groups` using `xarray`, you must use the `xr.open_datatree` [function](https://docs.xarray.dev/en/stable/generated/xarray.open_datatree.html) to open the data file. If you use the more familiar `xr.open_dataset` [function](https://docs.xarray.dev/en/stable/generated/xarray.open_dataset.html), you will **not** see the TEMPO data arrays!

[DataTree](https://docs.xarray.dev/en/stable/user-guide/data-structures.html#datatree), conventionally abbreviated as `dt`, is the highest-level data structure in `xarray` and is able to represent the recursive structure of multiple `groups` within a netCDF file.

We set the full directory path for the file (`pm25_file`) with the `pathlib` module, which uses `/` to join the path to the current working directory (`cwd`) and the file name.

In [ ]:
# Open TEMPO-ABI PM2.5 file using xarray DataTree
dt = xr.open_datatree(pm25_file, engine='netcdf4')

###**Read the metadata for the file & data variables**

The data are organized into three `Groups` in the file.

In each `group`, the satellite data are displayed under `Data variables`. For any of the `Data variables`, click on the attributes icon to see array metadata and the data repository icon to see an array summary.

A data variable in `xarray` is called a [DataArray](https://docs.xarray.dev/en/stable/generated/xarray.DataArray.html), conventionally abbreviated as `da`.

In [ ]:
# View metadata & contents of the entire file
dt

###**Read the satellite PM2.5 DataArrays**

The satellilte PM2.5 DataArrays are in the `product` group. There are separate variables for satellite PM2.5 geolocated on the [ABI fixed grid](https://www.star.nesdis.noaa.gov/atmospheric-composition-training/satellite_data_goes_imager_projection.php) for GOES-East (`pm25sat_ge`) and GOES-West (`pm25sat_gw`).

From the variable metadata, we see `pm25sat_ge` and `pm25sat_gw` have:
- Floating point number (`float32`) data type
- Units of micrograms per cubic meter ($\mu$g m${^{-3}}$)
- Valid range of 0 to 1000 $\mu$g m${^{-3}}$

The product User's Guide indicates there is **no** processing necessary for the satellite PM2.5 data, so we can use the DataArrays directly!

In [ ]:
# Read satellite PM2.5 DataArrays & assign to variables
sat_pm25_east = dt['product/pm25sat_ge']
sat_pm25_west = dt['product/pm25sat_gw']

### **View DataArrays as `NumPy` arrays**

Sometimes `xarray` does not display a summary of the array values in the metadata. When exploring a new satellite dataset, it's good practice to print a snippet of the DataArray to check the range of values.

Use the `xr.DataArray.values` [property](https://docs.xarray.dev/en/stable/generated/xarray.DataArray.values.html) to view the `sat_pm25_east` and `sat_pm25_west` DataArrays as `NumPy` arrays.

In [ ]:
# View DataArray as a NumPy array
sat_pm25_east.values

In [ ]:
# View DataArray as a NumPy array
sat_pm25_west.values

###**Find maximum and minimum values in the DataArrays, ignoring NaNs**

There are missing or invalid values at the beginning and end of the DataArrays; `xarray` automatically masks these values and displays them as `nan` (not a number).

To see the range of valid data in the arrays, use the `numpy.nanmax()` [function](https://numpy.org/doc/stable/reference/generated/numpy.nanmax.html) and `numpy.nanmin()` [function](https://numpy.org/doc/stable/reference/generated/numpy.nanmin.html) to find the non-nan maximum and minimum values.

In [ ]:
# Print non-NaN minimum and maximum values in DataArray
print(np.nanmin(sat_pm25_east.values), np.nanmax(sat_pm25_east.values))

In [ ]:
# Print non-NaN minimum and maximum values in DataArray
print(np.nanmin(sat_pm25_west.values), np.nanmax(sat_pm25_west.values))

### **Close the opened `xarray` DataTree**

Once you are finished with any opened files, it is best practice to close them.

In an actual script, it is cleaner and more pythonic to open files using the Python `with` [statement](https://www.geeksforgeeks.org/with-statement-in-python/), which automatically closes the file when you're done working with it. You can see an example of this approach in the code used to upload the satellite .nc files from my NOAA web archive to the Colab instance.

In [ ]:
# Close opened DataTree
dt.close()

##**Section 2: Latitude & Longitude for TEMPO-ABI PM2.5 Files**

###**Latitude/longitude grids for TEMPO-ABI PM2.5**

TEMPO-ABI hourly estimated surface PM2.5 data are gridded on the [ABI fixed grids](https://www.star.nesdis.noaa.gov/atmospheric-composition-training/satellite_data_goes_imager_projection.php#what_is_gip):
 - Locations east of 106 °W use the GOES-East ABI fixed grid
 - Locations west of 106 °W use the GOES-West ABI fixed grid

 The figure below illustrates these grids, using PM2.5 data (blue shading) from our case study file for May 14, 2026 at 19 UTC.

<div align="center">
<img src="https://raw.githubusercontent.com/NOAA-STAR/aerosolwatch-training/main/images/sat_pm25_grids.png" width="1000">
</div>



These grids do **not** vary. That means the latitude and longitude arrays of the TEMPO-ABI PM2.5 data do **not** vary from file to file.

As a result, it is usually more efficient to read in the latitude and longitude arrays once, from an external netCDF4 (.nc) file, instead of reading the arrays from each TEMPO-ABI PM2.5 data file that you use.

###**Why do some satellite lat/lon arrays contain missing values?**

Some geostationary satellite latitude and longitude arrays have cells that represent the view of deep space (not the Earth). Because there are no latitude or longitude values for space, these cells are assigned a `fill value` that `xarray` automatically masks as `nan` (not a number).

The figure below illustrates this phenomenon, using the GOES-East satellite's Full Disk view.  A 2D satellite data array can be conceptualized as a rectangle, outlined in black in the figure. The circle of the Earth's hemisphere lies inside the rectangle. All of the pixels represented by the Earth have valid `latitude` and `longitude` values. However, areas on the corners of the rectangle, shaded black, correspond to deep space and thus, they have no `latitude` or `longitude`. These deep space pixels are missing (masked) in the satellite data file.

<div align="center">
<img src="https://raw.githubusercontent.com/NOAA-STAR/aerosolwatch-training/main/images/goes_east_disk.png" width="400">
</div>

###**Upload interpolated latitude/longitude file for TEMPO-ABI PM2.5 data**

The TEMPO-ABI PM2.5 latitude and longitude arrays contain missing values. The `Matplotlib` `pyplot.pcolormesh` [plotting function](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.pcolormesh.html) is the preferred method to plot 2D satellite data. However, `pcolormesh` doesn't allow latitude or longitude arrays to have missing or masked values.

As a convenience for users, I made a netCDF4 file that contains the hourly satellite PM2.5 fixed grid latitude and longitude arrays, with the missing values interpolated, and posted the file on my NOAA public web archive. This file can be used with `pcolormesh` to plot the TEMPO-ABI PM2.5 data.

Let's upload this file to the Colab instance so we can use it for the tutorial.

In [ ]:
# Query NOAA web archive for satellite data files
data_url = ('https://www.star.nesdis.noaa.gov/pub/smcd/akhuff/'
            'Satellite_Lat_Lon_Files/'
            'hourly_satellite_pm25_interpolated_lat_lon.nc')
response = requests.get(data_url,
                        timeout=(5,20))

# Set directory path to file
lat_lon_file = Path.cwd() / 'hourly_satellite_pm25_interpolated_lat_lon.nc'

# Save file to Colab instance
with open(lat_lon_file, 'wb') as file:
    file.write(response.content)

###**Open latitude/longitude file as an `xarray` `Dataset`**

The lat/lon file does not use `groups`, so it can be opened as an `xarray` `Dataset` (`ds`) using the familiar `xr.open_dataset` [function](https://docs.xarray.dev/en/stable/generated/xarray.open_dataset.html).

In [ ]:
# Open lat/lon file using xarray Dataset
ds = xr.open_dataset(lat_lon_file, engine='netcdf4')

###**Read the metadata for the file & data variables**

The latitude and longitude data arrays are displayed under `Data variables`. Click on the attributes icon to see array metadata and the data repository icon to see an array summary.

In [ ]:
# View metadata & contents of the entire file
ds

###**Read the latitude and longitude DataArrays**

There are separate variables for the latitude and longitude derived from the ABI fixed grid for the GOES-East satellite (`_ge`) and GOES-West satellite (`_gw`).

From the variable metadata, we can see that these arrays contain floating point numbers (`float32`) with
 - units of `degrees_north` for `lat`
 - units of `degrees_east` for `lon`

In [ ]:
# Read latitude and longitude DataArrays & assign to variables
lon_east = ds.lon_ge
lat_east = ds.lat_ge
lon_west = ds.lon_gw
lat_west = ds.lat_gw

### **View DataArrays as ```NumPy``` arrays**

As before, we can use the `xr.DataArray.values` [property](https://docs.xarray.dev/en/stable/generated/xarray.DataArray.values.html) to view the DataArrays as `NumPy` arrays and print the contents.

In [ ]:
# View DataArray as a NumPy array
lon_east.values

In [ ]:
# View DataArray as a NumPy array
lat_east.values

In [ ]:
# View DataArray as a NumPy array
lon_west.values

In [ ]:
# View DataArray as a NumPy array
lat_west.values

###**Find maximum and minimum values in the DataArrays**

To see the range of valid data in the arrays and confirm there are no `nan` values, use the `numpy.max()` [function](https://numpy.org/doc/stable/reference/generated/numpy.max.html) and `numpy.min()` [function](https://numpy.org/doc/stable/reference/generated/numpy.min.html) to find the maximum and minimum values.

In [ ]:
# Print minimum and maximum values in DataArray
print(np.min(lon_east.values), np.max(lon_east.values))

In [ ]:
# Print minimum and maximum values in DataArray
print(np.min(lat_east.values), np.max(lat_east.values))

In [ ]:
# Print minimum and maximum values in DataArray
print(np.min(lon_west.values), np.max(lon_west.values))

In [ ]:
# Print minimum and maximum values in DataArray
print(np.min(lat_west.values), np.max(lat_west.values))

### **Close the opened `xarray` Dataset**

In [ ]:
# Close opened Dataset
ds.close()

##**Section 3: Get AirNow PM2.5 Monitor Data**

###**AirNow pollutant regulatory monitor data**

The [AirNow-Tech website](https://files.airnowtech.org/) provides files that contain near real-time and archived hourly and daily averaged pollutant concentrations from regulatory monitors reported to the AirNow network.

The `requests` package is used to access the AirNow-Tech website and extract the hourly PM2.5 monitor data file that corresponds to the observation time of the TEMPO-ABI PM2.5 data.

There is no need to download this AirNowTech data file! Instead, we **stream** the file using the `io` module and read the PM2.5 concentration, latitude, and longitude of the monitor site into a `pandas` [DataFrame](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.html) called `airnow_data`. When it is time to analyze or plot the PM2.5 monitor data, they can be read directly from the `airnow_data` `DataFrame`, as demonstrated in the next sections.

###**Query AirNow-Tech & Save Hourly AirNow PM2.5 to a `Dataframe`**

To navigate to the correct hourly pollutant data file on the AirNow-Tech website, its URL is built from strings of observation date and time, e.g. `'20260514'` and `'19'`. An easy way to obtain these strings automatically is to use the string of the TEMPO-ABI PM2.5 file name, obtained via the `.name` [method](https://docs.python.org/3/library/pathlib.html#pathlib.PurePath.name) of the `pathlib` module. Then the observation year, full date, and time are extracted using the Python `str.split()` [built-in type](https://docs.python.org/3/library/stdtypes.html#str.split) to split the file name using the `_` delimiter, in conjunction with Python [slicing](https://stackoverflow.com/questions/509211/how-slicing-in-python-works).

The `airnow_file` name and the `url` for the AirNow-Tech website are constructed using Python [formatted string literals](https://docs.python.org/3/tutorial/inputoutput.html#tut-f-strings) or `f-strings`. `f-strings` let you include the value of Python expressions (defined variables) inside a string by prefixing the string with `f` or `F` and writing expressions as `{expression}`. A long `f-string` should be split over multiple lines by placing parentheses around the entire `f-string`, enclosing each line in single or double quotes, and starting each line with `f` or `F`.

In [ ]:
# Return the final component of the file path as a string
pm25_file.name

In [ ]:
# Pull observation hour, year, and date strings from filename string
hour = pm25_file.name.split('_')[4][9:11]
year = pm25_file.name.split('_')[4][:4]
date = pm25_file.name.split('_')[4][:8]

In [ ]:
# Print the hour, year, and date strings (as a check)
print(hour)
print(year)
print(date)

In [ ]:
# Construct AirNow hourly filename
airnow_file = f'HourlyAQObs_{date}{hour}.dat'

# Query AirNow-Tech
url = (f'https://s3-us-west-1.amazonaws.com/'
       f'files.airnowtech.org/airnow/'
       f'{year}/{date}/')
response = requests.get(f'{url}{airnow_file}',
                        timeout=(5,20))

# Read text stream using in-memory text buffer
with StringIO(response.text) as response_text:
    columns = [4, 5, 22]
    airnow_data = pd.read_csv(response_text,
                              sep=',',
                              header=None,
                              usecols=columns,
                              names=['Lat', 'Lon', 'PM2.5'],
                              skiprows=1)

# Drop all rows with missing PM2.5 values
airnow_data.dropna(subset=['PM2.5'],
                    inplace=True)
# Reset DataFrame index
airnow_data.reset_index(inplace=True,
                        drop=True)

In [ ]:
# Print the first 5 rows of the Dataframe (as a check)
airnow_data.head()

##**Section 4: Generate ABI True Color RGB Composite using `SatPy`**

###**Generating true color RGB composites is complicated!**

It is very helpful to plot a true color (red-green-blue, RGB) composite image underneath satellite aerosol products, to see clouds and areas of smoke or blowing dust.

However, generating a satellite true color RGB array is a complicated, multi-step process that is far beyond the scope of this tutorial. A popular "shortcut" is to use the compositing features of the `SatPy` package.
- Advantage: You only need to download or stream the satellite input file(s); `SatPy` does everything else for you
- Disadvantage: `SatPy` is a large package with many dependencies, and the underlying steps in the RGB generation process are hidden

### **Upload ABI Multiband Cloud and Moisture Imagery Products (MCMIP) file**

To make a GOES-East ABI true color RGB image, we will use an ABI [Level 2](https://www.star.nesdis.noaa.gov/atmospheric-composition-training/satellite_data_processing_levels.php#level_2) Multichannel Cloud & Moisture Imagery Products [CONUS sector](https://www.star.nesdis.noaa.gov/atmospheric-composition-training/satellite_data_abi_scan_sectors.php#conus) (ABI-L2-MCMIPC) product file.

MCMIP files are easy to work with because they contain data from all 16 ABI channels at a consistent spatial resolution of 2km: dimensionless reflectance (normalized by the solar zenith angle) for the ABI reflective bands (1-6) and brightness temperature (BT) at the Top-Of-Atmosphere (TOA) in Kelvin for the ABI emissive bands (7-16).

For convenience for this tutorial, the GOES-19 (GOES-East) ABI-L2-MCMIPC data file for May 14, 2026 at 19:31 UTC can be uploaded from my NOAA public web archive.

In [ ]:
# Query NOAA web archive for satellite data files
data_url = ('https://www.star.nesdis.noaa.gov/pub/smcd/akhuff/'
            'Tutorial_Satellite_Data_Files/'
            'OR_ABI-L2-MCMIPC-M6_G19_s20261341931172_e20261341933546_c20261341934231.nc')
response = requests.get(data_url,
                        timeout=(5,20))

# Set directory path to file
abi_mcmip_file = Path.cwd() / 'OR_ABI-L2-MCMIPC-M6_G19_s20261341931172_e20261341933546_c20261341934231.nc'

# Save file to Colab instance
with open(abi_mcmip_file, 'wb') as file:
    file.write(response.content)

### **Generate ABI true color RGB array using `SatPy`**

Run the code below to load the ABI MCMIP file in the `SatPy` true color RGB `reader` and get:
- RGB composite array generated by `SatPy` as a `NumPy` array (`rgb_arr`)
- Map projection `SatPy` used to resample the ABI bands (`crs`)

Both of these variables are needed to plot the RGB composite on a map, as demonstrated in the next section.

**NOTE: Ignore any warning messages from `PySpectral`!**

In [ ]:
# Load ABI MCMIP data file to Satpy reader for true color RGB
scn = Scene(reader='abi_l2_nc',
            filenames=[abi_mcmip_file])
scn.load(['true_color'])

# Resample loaded bands to prepare for compositing
resamp = scn.resample(scn.coarsest_area(),
                      resampler='native')

# Get numpy array of RGB composite
rgb_da = get_enhanced_image(resamp['true_color']).data
rgb_arr = rgb_da.transpose('y', 'x', 'bands').values.clip(0, 1)

# Get projection of RGB array from Satpy
crs = resamp['true_color'].attrs['area'].to_cartopy_crs()

##**Section 5: Plot TEMPO-ABI PM2.5 on a Map**

###**Create color map for PM2.5 concentration using `Matplotlib`**

A suitable **colormap** is required to visualize data plotted on a map. The `Matplotlib` package has many [built-in colormaps](https://matplotlib.org/stable/tutorials/colors/colormaps.html).

For PM2.5 concentration, we can make a discrete color map that corresonds to the [US EPA's Air Quality Index (AQI)](https://www.airnow.gov/aqi/aqi-basics/) breakpoints for PM2.5, using Matplotlib [built-in colors](https://matplotlib.org/stable/gallery/color/named_colors.html).

In [ ]:
# Create discrete colormap based on US EPA's PM2.5 AQI
pm25_cmap = mpl.colors.ListedColormap(['limegreen',
                                      'yellow',
                                      'orange',
                                      'red',
                                      'darkorchid',
                                      'firebrick'])
bounds = [0, 9.1, 35.5, 55.5, 125.5, 225.5, 500]
cmap_norm = mpl.colors.BoundaryNorm(bounds, pm25_cmap.N)

### **Plot AirNow PM2.5 on a map with ABI RGB background**

To see the blowing dust plume, let's first plot the ABI true color RGB composite for 19:31 UTC with the 1-hour average AirNow PM2.5 for 19:00-19:59 UTC.

**Ignore the `DownloadWarning` messages**; these will appear the first time [Natural Earth Feature shapefiles](https://cartopy.readthedocs.io/stable/reference/generated/cartopy.io.shapereader.natural_earth.html#cartopy.io.shapereader.natural_earth) (used to format the map background) are downloaded via the `cartopy` `Feature` interface.

In [ ]:
# Plot hourly AirNow PM2.5 on SatPy ABI RGB composite

# Set up figure in matplotlib
fig = plt.figure(figsize=(8, 10))

# Set map projection using cartopy
ax = plt.axes(projection=ccrs.PlateCarree())

# Set geographic domain of map
# °E longitude > 0 > °W longitude, °N latitude > 0 > °S latitude
map_domain = [-112.5, -95.5, 42, 52]
ax.set_extent(map_domain,
              crs=ccrs.PlateCarree())

# Format map background
ax.add_feature(cfeature.COASTLINE,
               linewidth=0.5,
               zorder=3)
ax.add_feature(cfeature.BORDERS,
               linewidth=0.5,
               zorder=3)
ax.add_feature(cfeature.STATES,
               linewidth=0.5,
               zorder=3)
ax.add_feature(cfeature.LAKES,
               facecolor='none',
               edgecolor='black',
               linewidth=0.5,
               zorder=3)

# Plot SatPy ABI RGB composite
ax.imshow(rgb_arr,
          origin='upper',
          extent=crs.bounds,
          transform=crs,
          zorder=2)

# Plot AirNow PM2.5
ax.scatter(airnow_data['Lon'],
           airnow_data['Lat'],
           c=airnow_data['PM2.5'],
           s=5**2,
           linewidths=0.25,
           edgecolors='black',
           cmap=pm25_cmap,
           norm=cmap_norm,
           transform=ccrs.PlateCarree(),
           zorder=3)

# Add & format colorbar
cb = fig.colorbar(plt.cm.ScalarMappable(norm=cmap_norm,
                                        cmap=pm25_cmap),
                  ax=ax,
                  orientation='horizontal',
                  fraction=0.25,
                  pad=0.01,
                  shrink=0.5,
                  ticks=bounds,
                  spacing='uniform',
                  extend='max')
cb.set_label(label=r'PM$_{2.5}$ Concentration ($\mu$g m$\mathregular{^{-3}}$)',
             size=8,
             weight='bold')
cb.ax.set_xticklabels(['0', '9.1', '35.5', '55.5', '125.5', '225.5', '500'])
cb.ax.tick_params(labelsize=8)

# Show plot
plt.show()

# Save image file to Colab instance
# Use previously defined strings from sat file name
save_name = f'airnow_pm25_rgb_{date}_{hour}'
fig.savefig(Path.cwd() / save_name,
            dpi=300,
            bbox_inches='tight')

# Release memory
plt.close(fig)

###**Plot TEMPO-ABI & AirNow PM2.5 on a map with ABI RGB background**

Finally, let's add the TEMPO-ABI PM2.5 to the map. The code below is the same as the code in the previous block, with the exception of the addtion of two `pcolormesh` plotting lines for the TEMPO-ABI PM2.5 data on the GOES-East and -West grids.

**Note:** The plots in this section use the latitude, longitude, and satellite PM2.5 variables we defined in Sections 1 & 2. When you work with the data files on your own, these steps are not necessary. You can open the satellite data files using a `with` statement and plot the TEMPO-ABI data using the `dt['product/pm25sat_ge']`, `dt['product/pm25sat_gw']`, `ds.lon_ge`, `ds.lat_ge`, `ds.lon_gw`, and `ds.lat_gw` `xarray` DataArrays directly in the `pcolormesh` plotting lines.

In [ ]:
# Plot hourly TEMPO-ABI & AirNow PM2.5 on SatPy ABI RGB composite

# Set up figure in matplotlib
fig = plt.figure(figsize=(8, 10))

# Set map projection using cartopy
ax = plt.axes(projection=ccrs.PlateCarree())

# Set geographic domain of map
# °E longitude > 0 > °W longitude, °N latitude > 0 > °S latitude
map_domain = [-112.5, -95.5, 42, 52]
ax.set_extent(map_domain,
              crs=ccrs.PlateCarree())

# Format map background
ax.add_feature(cfeature.COASTLINE,
               linewidth=0.5,
               zorder=3)
ax.add_feature(cfeature.BORDERS,
               linewidth=0.5,
               zorder=3)
ax.add_feature(cfeature.STATES,
               linewidth=0.5,
               zorder=3)
ax.add_feature(cfeature.LAKES,
               facecolor='none',
               edgecolor='black',
               linewidth=0.5,
               zorder=3)

# Plot SatPy ABI RGB composite
ax.imshow(rgb_arr,
          origin='upper',
          extent=crs.bounds,
          transform=crs,
          zorder=2)

# Plot TEMPO-ABI PM2.5 (East & West grids)
ax.pcolormesh(lon_east,
              lat_east,
              sat_pm25_east,
              cmap=pm25_cmap,
              norm=cmap_norm,
              transform=ccrs.PlateCarree(),
              zorder=2.5)
ax.pcolormesh(lon_west,
              lat_west,
              sat_pm25_west,
              cmap=pm25_cmap,
              norm=cmap_norm,
              transform=ccrs.PlateCarree(),
              zorder=2.5)

# Plot AirNow PM2.5
ax.scatter(airnow_data['Lon'],
           airnow_data['Lat'],
           c=airnow_data['PM2.5'],
           s=5**2,
           linewidths=0.25,
           edgecolors='black',
           cmap=pm25_cmap,
           norm=cmap_norm,
           transform=ccrs.PlateCarree(),
           zorder=3)

# Add & format colorbar
cb = fig.colorbar(plt.cm.ScalarMappable(norm=cmap_norm,
                                        cmap=pm25_cmap),
                  ax=ax,
                  orientation='horizontal',
                  fraction=0.25,
                  pad=0.01,
                  shrink=0.5,
                  ticks=bounds,
                  spacing='uniform',
                  extend='max')
cb.set_label(label=r'PM$_{2.5}$ Concentration ($\mu$g m$\mathregular{^{-3}}$)',
             size=8,
             weight='bold')
cb.ax.set_xticklabels(['0', '9.1', '35.5', '55.5', '125.5', '225.5', '500'])
cb.ax.tick_params(labelsize=8)

# Show plot
plt.show()

# Save image file to Colab instance
# Use previously defined strings from sat file name
save_name = f'tempo-abi_airnow_pm25_rgb_{date}_{hour}'
fig.savefig(Path.cwd() / save_name,
            dpi=300,
            bbox_inches='tight')

# Release memory
plt.close(fig)

###**Don't forget: download files you want to keep!**

Remember, any files you want to keep must be downloaded from the Colab instance to your Google Drive or local computer.

Using the controls in the `Files` menu panel on the left side of the Colab window, you can manually mount your Google Drive to the Colab instance or manually download files to your local computer.

##**Appendix: Details about the Data Plotting Code**

The map projection is set using the `cartopy` package. There are many [map projection options](https://cartopy.readthedocs.io/stable/reference/projections.html#cartopy-projections) available from `cartopy`. I used the [Plate Carree](https://cartopy.readthedocs.io/stable/reference/projections.html#cartopy.crs.PlateCarree) rectangular equidistant projection.

 The `cartopy` `set_extent([bounding_box_corners])` [function](https://cartopy.readthedocs.io/stable/reference/generated/cartopy.mpl.geoaxes.GeoAxes.html#cartopy.mpl.geoaxes.GeoAxes.set_extent) sets the geographic domain of the map, where the `bounding_box_corners` are the `[western_longitude, eastern_longitude, southern_latitude, northern_latitude]` of the zoomed-in domain in degrees.
 - The argument `crs=ccrs.PlateCarree()` must be used with `set_extent()`  because the `bounding_box_corners` are entered in geographic coordinates (latitude/longitude).

The `cartopy` [Feature interface](https://cartopy.readthedocs.io/stable/reference/feature.html) is used to add coastlines and borderlines.
 - Ignore the `DownloadWarning` messages; these will appear the first time [Natural Earth Feature shapefiles](https://cartopy.readthedocs.io/stable/reference/generated/cartopy.io.shapereader.natural_earth.html#cartopy.io.shapereader.natural_earth) are downloaded via the `cartopy` `Feature` interface.

The ABI RGB composite is plotted using the `matplotlib.pyplot.imshow()` [plotting function](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.imshow.html). The arguments `extent=crs.bounds` and `transform=crs` must be used or the RGB composite will not plot correctly! The `crs` is the projection derived from `SatPy` in Section 4. It is the projection that `Satpy` used to resample the ABI-L2-MCMIPC data.

The argument `transform=ccrs.PlateCarree()` must be used in the `matplotlib.pyplot.pcolormesh()` [plotting function](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.pcolormesh.html) because the satellite data are in geographic coordinates (latitude/longitude).  **Failure to set this argument to `ccrs.PlateCarree()` will result in plotting errors!**

The `zorder=` [argument](https://matplotlib.org/stable/gallery/misc/zorder_demo.html) sets the order for plotting layers, so the borders/coastlines plot on top of the TEMPO-ABI PM2.5, which plots on top of the RGB composite.

A colorbar is added and formatted using the `matplotlib.colorbar.colorbar()` [function](https://matplotlib.org/stable/api/colorbar_api.html).

The map image files are saved to the Colab instance (under `Files` in the menu panel on the left side of the Colab window) using the the `matplotlib` `fig.savefig()` [function](https://matplotlib.org/stable/api/figure_api.html#matplotlib.figure.Figure.savefig).
- The `dpi=` argument sets the resolution of the saved image file. The higher the dpi, the higher the figure resolution, but the larger the file size and the longer it will take to save the file. The default is  `dpi=100`. Try `dpi=300` for general use, `dpi=600` for presentations, and `dpi=1000` for journal articles.
- The `bbox_inches='tight'` argument minimizes the bounding box around the figure (zooms in "tight" on the plot).